# Forward Pass Debugging: Layer-by-Layer Reference

This notebook performs a forward pass through a .keras model one layer at a time, for the purpose of verifying low-level implementations in C or RISC-V.

At each step, it prints the output of the current layer. These values act as ground truth for validating manual implementations. The output of one layer is passed directly as the input to the next, preserving the inference flow.

This setup helps identify discrepancies between the high-level model and its low-level counterparts, allowing for precise, layer-specific debugging.


In [ ]:
import os
import tensorflow as tf
import numpy as np
import random

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


## 1.0 Load and preprocess the data

In [22]:
(images, labels), _ = tf.keras.datasets.mnist.load_data()
images = images.astype("float32") / 255.0  # Normalize the images to [0, 1]
images = np.expand_dims(images, -1)  # Add channel dimension
labels = tf.keras.utils.to_categorical(labels, 10)  # One-hot encode the labels

## 2.0 Helper Functions

### 2.1 Function to Get a random image and label

Get a random image and label from the dataset. This function is used to generate a random input for the model.

In [23]:
def get_random_image(output_file="random_image.txt"):
    # Randomly select an image and its label
    index = random.randint(0, len(images) - 1)
    image = images[index].squeeze()  # (28, 28)
    image = tf.expand_dims(image, axis=0)  # Add batch dimension (1, 28, 28)
    image = tf.expand_dims(image, axis=-1)  # Add channel dimension (1, 28, 28, 1)
    
    label = np.argmax(labels[index])  # Get label
    
    # Save the image to a .txt file with 3 decimal places
    with open(output_file, "w") as f:
        for row in image.numpy().squeeze():  # Convert tensor to numpy and remove extra dimensions
            row_str = ".float " + ", ".join(f"{val:.3f}" for val in row)  # Using commas to separate values
            f.write(row_str + "\n")
    
    return image, label

### 2.2 Function to Print a tensor

Prints the shape and values of a tensor in a readable format.

For 4D tensors (e.g., batches of images), it prints the values of the first sample,
channel by channel. For 2D tensors (e.g., dense layer outputs), it prints all values
row by row. Other shapes are not currently supported.

In [33]:
def print_shape_and_values(x):
    print(f"Shape: {x.shape}")
    
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                row = [f"{x[0, i, j, c]:.3f}" for j in range(width)]
                print(", ".join(row))
            print()
    
    elif len(x.shape) == 2:
        rows, cols = x.shape
        for i in range(rows):
            row = [f"{x[i, j]:.3f}" for j in range(cols)]
            print(", ".join(row))
    else:
        print("Unsupported shape")


## 3.0 Load the model from mnist_cnn_model.keras

In [34]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

## 4.0 Get a random image, label and step through the model layer by layer

### 4.1 Get a random image and label

In [35]:
image, label = get_random_image()
print(f"Label: {label}")

Label: 1


### 4.2 Step through the model layer by layer

#### 4.2.1 Input Image

In [36]:
print("Original Image:")
print_shape_and_values(image)

Original Image:
Shape: (1, 28, 28, 1)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.251, 1.000, 1.000, 0.251, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 

#### 4.2.2 Conv2D Layer

Note: Refer to section **4.2.1** for the input

In [37]:
conv2d_out = model.layers[0](image)
print_shape_and_values(conv2d_out)

Shape: (1, 24, 24, 8)
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.398, -0.164, -0.034, -0.279, -0.505, -0.551, -0.538, -0.480, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.258, 0.008, 0.023, -0.006, -0.090, -0.149, -0.315, -0.459, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.398, -0.142, -0.146, -0.251, -0.098, 0.059, 0.076, -0.172, -0.406, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.319, -0.206, -0.498, -0.451, 0.002, 0.167, -0.034, -0.318, -0.444, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.460, -0.398, -0.161, -0.323, -0.489, -0.233, -0.010, 0.074, -0.052, -0.459, -0.508, -0.460, -0.460, -0.460, -0.460
-0.460, -0.460, -0.460, -0.460

#### 4.2.3 ReLU Activation

Note: Refer to section **4.2.2** for the input

In [38]:
relu_out = model.layers[1](conv2d_out)
print_shape_and_values(relu_out)

Shape: (1, 24, 24, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.008, 0.023, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.059, 0.076, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.002, 0.167, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.074, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.055, 0.000, 0.000, 0.000, 0.000, 0.0

#### 4.2.4 MaxPooling

Note: Refer to section **4.2.3** for the input

In [39]:
maxpool_out = model.layers[2](relu_out)
print_shape_and_values(maxpool_out)

Shape: (1, 12, 12, 8)
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.008, 0.023, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.002, 0.167, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.055, 0.074, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.084, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.137, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.201, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.002, 0.026, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.092, 0.030, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.021, 0.282, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.084, 0.050, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.198, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000
0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.

#### 4.2.5 Flatten

Note: Refer to section **4.2.4** for the input

In [40]:
flatten_out = model.layers[3](maxpool_out)
print_shape_and_values(flatten_out)

Shape: (1, 1152)
0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.008, 1.127, 0.000, 0.000, 0.000, 0.588, 0.190, 0.000, 0.023, 2.185, 0.000, 0.000, 1.015, 0.524, 1.101, 0.000, 0.000, 2.298, 0.626, 0.330, 0.554, 0.000, 0.272, 0.000, 0.000, 1.273, 0.378, 0.672, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.178, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.414, 0.000, 0.000, 0.0

#### 4.2.6 Fifth Layer: Dense Layer

Note: Refer to section **4.2.5** for the input

In [41]:
dense_out = model.layers[4](flatten_out)  # Fifth layer output
print_shape_and_values(dense_out)

Shape: (1, 10)
-4.271, 9.697, -1.265, -6.898, -1.932, -7.214, -8.857, -6.193, -2.045, -10.366


#### 4.2.7 Layer Six: Softmax

Note: Refer to section **4.2.6** for the input

In [42]:
softmax_out = model.layers[5](dense_out)
print_shape_and_values(softmax_out)

Shape: (1, 10)
0.000, 1.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000, 0.000


### 4.3 Get model prediction

In [43]:
print(f"Predicted class: {np.argmax(softmax_out)}")
print(f"True class: {label}")

Predicted class: 1
True class: 1
